# Qwen3.5 Image-to-SMT via Grammar-Constrained Decoding

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import outlines
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    DATA_DIR,
    CVC5_PATH,
    PREAMBLE,
    PROMPT_TEMPLATE_PASS1A,
    PROMPT_TEMPLATE_PASS1B,
    PROMPT_TEMPLATE_PASS2,
    PROMPT_TEMPLATE_PLANNING,
    PROMPT_TEMPLATE_REFLECTION,
    PROMPTS,
    SMT_LIB_GRAMMAR_PASS1A,
    EXAMPLES_PASS1A,
    EXAMPLES_PASS1B,
    EXAMPLES_PASS2,
    SMT_MAX_NEW_TOKENS,
    SMT_TEMPERATURE,
    SMT_TOP_P,
    SMT_TOP_K,
    SMT_MIN_P,
    SMT_PRESENCE_PENALTY,
    SMT_REPETITION_PENALTY,
)
from staged_qwen3_5_scivqa.context import get_paper_context
from staged_qwen3_5_scivqa.smt.solver import validate_smt
from staged_qwen3_5_scivqa.smt.pipeline import (
    clean_duplicate_declarations,
    deduplicate_anchors,
    generate_declarations,
    parse_table_deterministically,
    reflect,
)
from staged_qwen3_5_scivqa.smt.grammars import (
    SMT_CFG_PASS1A,
    build_dynamic_phase1b_cfg,
    build_dynamic_phase2_cfg,
)

In [ ]:
# MODEL_ID = "unsloth/Qwen3.5-35B-A3B"

MODEL_ID = "unsloth/Qwen3.5-9B"
MAX_NEW_TOKENS = 2048

DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

STATE_FILE = DATA_DIR / f"smt_{CATEGORY}_state.json"
SMT_FILE = DATA_DIR / f"smt_{CATEGORY}.json"

SUMMARY_CACHE_PATH = DATA_DIR / f"submission_finetuning_summary_{CATEGORY}_state.json"
EXTRACTION_CACHE_PATH = (
    DATA_DIR / f"submission_finetuning_extraction_{CATEGORY}_state.json"
)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen3.5 (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

### SMT Pipeline Functions (imported from library)

### Model Loading



In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use torch.float16 if your GPU is older and doesn't support bfloat16
    bnb_4bit_use_double_quant=True,  # Optional: Saves a bit more memory
    bnb_4bit_quant_type="nf4",  # Optional: Recommended for better performance
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

lm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb_config
)

In [ ]:
tokenizer.chat_template = (
    "{% set enable_thinking = false %}\n" + tokenizer.chat_template
)

In [ ]:
model = outlines.from_transformers(lm, tokenizer)

In [ ]:
if STATE_FILE.exists():
    with open(STATE_FILE) as f:
        state = json.load(f)
    print(f"Loaded existing state with {len(state)} samples.")
else:
    state = {}
    print("Initialized empty state.")

In [ ]:
summary_cache = None
with open(SUMMARY_CACHE_PATH) as f:
    summary_cache = json.load(f)

extraction_cache = None
with open(EXTRACTION_CACHE_PATH) as f:
    extraction_cache = json.load(f)

split_dir = COMPETITION_DATA_DIR / CATEGORY
assert split_dir.exists(), f"Directory for split '{CATEGORY}' not found!"

json_files = list(split_dir.rglob("*.json"))

processed_cnt = 0
skipped_cnt = 0
already_processed_cnt = 0

# ---------------------------------------------------------
# PASS 1: Scan and flatten the workload (takes just a moment)
# ---------------------------------------------------------
tasks = []
print(f"Scanning {CATEGORY} files to calculate actual workload...")

for json_file in json_files:
    fullpath = str(json_file)

    # File-level skips
    if (
        "content.json" in json_file.name
        or "images" not in fullpath
        or ".vscode" in fullpath
    ):
        continue

    img_path = json_file.with_suffix(".jpg")
    if not img_path.exists():
        warnings.warn(f"Skipping JSON file (missing image {img_path.name}): {fullpath}")
        continue

    with open(json_file) as f:
        data = json.load(f)

    sample_id = data.get("sample_id")
    if not sample_id:
        warnings.warn(f"Skipping JSON file (missing 'sample_id'): {fullpath}")
        continue

    if sample_id not in state:
        state[sample_id] = {}

    bboxes = data.get("bbox", {})
    vqa_data = data.get("vqa", {})

    # Gather valid questions
    for sub_key, q_list in vqa_data.items():
        if sub_key not in state[sample_id]:
            state[sample_id][sub_key] = {}

        table = extraction_cache.get(sample_id, {}).get(sub_key, None)

        # Subfigure-level skips
        if not table:
            warnings.warn(
                f"Skipping subfigure '{sub_key}' in {sample_id} (no table data)."
            )
            skipped_cnt += len(q_list) if isinstance(q_list, list) else 1
            continue

        if sub_key not in bboxes:
            warnings.warn(
                f"Skipping subfigure '{sub_key}' in {sample_id} (no bounding box)."
            )
            skipped_cnt += len(q_list) if isinstance(q_list, list) else 1
            continue

        for q_obj in q_list:
            question_text = q_obj.get("question") or q_obj.get("questions")

            # Question-level skips
            if not question_text:
                warnings.warn(
                    f"Skipping a question in {sample_id} subfigure '{sub_key}' (missing question text)."
                )
                skipped_cnt += 1
                continue

            if question_text in state[sample_id][sub_key]:
                already_processed_cnt += 1
                continue

            # If it passes all checks, it's a valid task!
            tasks.append(
                {
                    "json_file": json_file,
                    "img_path": img_path,
                    "sample_id": sample_id,
                    "sub_key": sub_key,
                    "q_obj": q_obj,
                    "question_text": question_text,
                    "box": bboxes[sub_key],
                    "summary": summary_cache.get(sample_id, {}).get(sub_key, ""),
                    "table": table,
                }
            )

# ---------------------------------------------------------
# PASS 2: Process the flattened tasks with an accurate tqdm
# ---------------------------------------------------------
pbar = tqdm(tasks, desc=f"Processing {CATEGORY} split", position=1, leave=False)

# Track the current image so we don't repeatedly load the same image from disk
current_img_path = None
full_img = None

for task in pbar:
    sample_id = task["sample_id"]
    sub_key = task["sub_key"]

    pbar.set_description(f"Reflecting: {sample_id} | Sub: {sub_key}")

    # Only open a new image if the file path has changed
    if current_img_path != task["img_path"]:
        current_img_path = task["img_path"]
        full_img = Image.open(current_img_path.absolute())

    # Crop image
    box = task["box"]
    left, top = box["x"], box["y"]
    right, bottom = left + box["width"], top + box["height"]
    crop = full_img.crop((left, top, right, bottom))

    # Run reflection
    smt_code, solver_output = reflect(
        model=model,
        q_obj=task["q_obj"],
        image=crop,
        summary=task["summary"],
        table=task["table"],
        max_retries=1,
        verbose=False,
        do_sample=True,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=SMT_TEMPERATURE,
        min_p=SMT_MIN_P,
        top_p=SMT_TOP_P,
        top_k=SMT_TOP_K,
        repetition_penalty=SMT_REPETITION_PENALTY,
    )

    # Save to state
    state[sample_id][sub_key][task["question_text"]] = {
        "code": smt_code,
        "output": solver_output,
    }

    # Update counters
    processed_cnt += 1

    # Checkpoint every 5 tasks to save disk I/O time
    if processed_cnt % 5 == 0:
        with open(STATE_FILE, "w") as f:
            json.dump(state, f, indent=4)

    pbar.set_postfix(
        processed=processed_cnt, skipped=skipped_cnt, already=already_processed_cnt
    )

print(
    f"Pipeline execution complete! State fully synced.\n"
    f"Summary -> Processed: {processed_cnt} | Already Processed: {already_processed_cnt} | Skipped: {skipped_cnt}"
)

In [ ]:
print(f"Saving final consolidated data to {SMT_FILE}...")
with open(SMT_FILE, "w") as f:
    json.dump(state, f, indent=4)

# Delete the temporary state file now that the final write is complete
if STATE_FILE.exists():
    STATE_FILE.unlink()
    print(f"Deleted temporary state file: {STATE_FILE}")

print("All done! Data successfully written to SMT_FILE.")